# 20261109 · 课程串讲：从 0 搭一条 RAG 流水线

用《峨眉山讲解词.docx》搭一个景区问答助手。建库时把文档变成可检索的证据片段。问答时取回相关片段，把正文和出处一起交给模型。

课程材料在 `week25/20261109RAG/`。`RAG1.ipynb` 讲概念，`RAG2.ipynb` 演示流程，`test1.py`～`test5.py` 是加载、清洗、切块、嵌入、建库的练习。

| 之前的课程 | 本讲的对应位置 |
|---|---|
| LangChain | 第 2～8 节用加载器、切块器、向量库和模型串起问答 |
| 工具调用 / LangGraph | 第 9 节把检索器作为工具交给 Agent，观察模型选用工具和改写查询 |
| SQL Agent | 两者都先取得外部资料。SQL 查询结构化行，本讲从向量库取原文片段 |

## 执行说明

- 使用 `week23` 的 Python 3.11 环境。从 `week25` 启动 Notebook，或在第 0 节填写 `DATA_ROOT`。
- 默认 `RUN_LIVE = False`，可离线运行加载、清洗、切块和评测数据校验。打开 `RUN_LIVE` 后，嵌入请求会将待向量化的正文和问题发给云端接口，生成请求也会发送检索到的正文。第 10 节的对比实验另由 `RUN_EVAL` 开启。
- 云端模式使用 `week23/.env` 的千问配置。建库演示每次在 `chroma_db/` 下建立独立目录。
- 本讲主语料是 Word 文档。第 2.4 节用临时生成的两页 PDF 演示按页加载。课堂里的 Unstructured 和 jionlp 示例用于对照。

## 0. 环境自检

先定位数据、检查依赖，再按需探测云端。离线模式可以独立学习文档预处理。打开 `RUN_LIVE` 后，本讲使用 `lab_config.get_model()` 创建千问对话模型，并分别发送一条短文本的嵌入请求和对话请求。嵌入模型在本讲显式指定为 `text-embedding-v4`，输出 1024 维，每批最多 10 条。

In [1]:
import importlib.metadata as md
import math
import re
import sys
import time
from pathlib import Path
from tempfile import TemporaryDirectory, mkdtemp

import numpy as np

RUN_LIVE = False
RUN_EVAL = False
DATA_ROOT = None  # 从其他目录启动时，可填 week25 或 20261109RAG 的路径

start_dir = Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else Path.cwd()
docx_candidates = [
    start_dir / "20261109RAG" / "峨眉山讲解词.docx",
    start_dir / "峨眉山讲解词.docx",
]
DOCX = next((path for path in docx_candidates if path.is_file()), None)
if DOCX is None:
    raise FileNotFoundError("找不到《峨眉山讲解词.docx》。请把 DATA_ROOT 指向 week25 或 20261109RAG。")

NOTEBOOK_DIR = DOCX.parent.parent if DOCX.parent.name == "20261109RAG" else DOCX.parent
DB_DIR = NOTEBOOK_DIR / "chroma_db"
EMBED_MODEL = "text-embedding-v4"
EMBED_DIM = 1024
embedding_model = llm = None

print("== 包版本 ==")
for name in ("langchain", "langchain-core", "langchain-community", "langchain-text-splitters",
             "langchain-openai", "langchain-chroma", "chromadb", "pymupdf", "docx2txt",
             "python-docx", "clean-text", "tiktoken"):
    try:
        print(f"  {name:26s}{md.version(name)}")
    except md.PackageNotFoundError:
        print(f"  {name:26s}[未安装]")
print(f"资料：{DOCX.name}（{DOCX.stat().st_size / 1024:.0f} KB）")

if RUN_LIVE:
    from langchain_openai import OpenAIEmbeddings

    lab_config_dir = NOTEBOOK_DIR.parent / "week23" / "20261027"
    if not (lab_config_dir / "lab_config.py").is_file():
        raise FileNotFoundError(f"找不到模型配置：{lab_config_dir / 'lab_config.py'}")
    if str(lab_config_dir) not in sys.path:
        sys.path.insert(0, str(lab_config_dir))
    import lab_config

    if lab_config.PROVIDER != "qwen":
        raise ValueError("本讲云端示例需要在 week23/.env 设置 LLM_PROVIDER=qwen。")
    llm = lab_config.get_model(temperature=0)
    embedding_model = OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=lab_config.BASE_URL,
        api_key=lab_config.API_KEY,
        check_embedding_ctx_length=False,
        dimensions=EMBED_DIM,
        chunk_size=10,
    )
    probe_vector = embedding_model.embed_query("嵌入连通测试")
    if len(probe_vector) != EMBED_DIM or not np.isfinite(probe_vector).all():
        raise ValueError("嵌入接口返回了错误的维度或非有限数值。")
    if not llm.invoke("请只回复 OK。").content:
        raise RuntimeError("对话接口正文为空，请检查模型配置和输出预算。")
    print(f"云端探测通过：对话 {lab_config.MODEL}，嵌入 {EMBED_MODEL} / {len(probe_vector)} 维")
else:
    print("离线模式：跳过云端模型初始化。")

== 包版本 ==
  langchain                 1.4.7
  langchain-core            1.7.0
  langchain-community       0.4.2
  langchain-text-splitters  1.1.3
  langchain-openai          1.7.3
  langchain-chroma          1.1.0
  chromadb                  1.5.9
  pymupdf                   2.1
  docx2txt                  0.9
  python-docx               1.2.0
  clean-text                0.7.1
  tiktoken                  0.14.0
资料：峨眉山讲解词.docx（156 KB）
离线模式：跳过云端模型初始化。


## 1. RAG 到底解决什么

| 场景 | RAG 能提供什么 | 系统需要做什么 |
|---|---|---|
| 资料会更新 | 回答可以依据新版本资料 | 更新文档和索引，清理过时片段 |
| 回答需要核验 | 把支持结论的原文和出处放进上下文 | 保留来源，检查引用是否支持结论 |
| 有专门的业务资料 | 在已有模型上接入自己的文档 | 明确资料访问范围和处理位置 |

检索器拿到候选片段，片段进入 Prompt，大模型生成答案。这条链路的质量取决于检索结果能否提供足够证据，以及模型是否正确使用这些证据。

### 七步流水线（本讲的目录）

```mermaid
flowchart LR
    subgraph OFF[入库阶段：首次建库或资料更新]
        A[原始文档<br/>docx / pdf] --> B[1 加载<br/>Document + metadata]
        B --> C[2 清洗<br/>修排版 / 去噪]
        C --> D[3 切块<br/>chunk_size + overlap]
        D --> E[4 嵌入<br/>文本 → 1024 维]
        E --> F[(5 向量库<br/>Chroma)]
    end
    subgraph ON[问答阶段：每次请求]
        Q[用户问题] --> QE[问题嵌入] --> R[6 检索 top-k]
        F -.-> R
        R --> P[拼 Prompt<br/>资料 + 问题] --> L[7 生成] --> ANS[带出处的答案]
    end
```

| # | 步骤 | 本节 | 课堂位置 |
|---|---|---|---|
| 1 | 加载 | 第 2 节 | `RAG2` 第 4~10 格 / `test1.py` |
| 2 | 清洗 | 第 3 节 | `RAG2` 数据清洗一节 / `test2.py` |
| 3 | 切块 | 第 4 节 | `RAG2` 文本分割器 / `test3.py` |
| 4 | 嵌入 | 第 5 节 | `RAG2` 文本词嵌入 / `test4.py` |
| 5 | 入库 | 第 6 节 | `RAG2` 向量数据库 / `test5.py` |
| 6 | 检索 | 第 7 节 | `RAG2` 检索器一节 |
| 7 | 生成 | 第 8 节 | `RAG2` 检索器转工具之前 |

### 做一次无资料基线

一问指定了讲解词里的万年寺细节，一问是公开的地理信息。第 8 节会保留同一问题、同一模型和同一系统提示，只在用户消息里加入检索资料，观察答案内容和出处的变化。

In [2]:
blind_questions = [
    "《峨眉山导游词》里说，万年寺的砖殿为什么叫「无梁砖殿」？殿里还有什么宝贝？",
    "峨眉山最高峰叫什么？海拔多少米？",
]
SYSTEM_PROMPT = (
    "你是峨眉山景区讲解助手。"
    "如果消息提供了【资料】，只用资料作答，每条结论后用 [编号] 引用支持它的片段。"
    "资料不足时，回答「资料中没有相关信息」。"
    "如果消息只有【问题】，给出简短回答，并明确说明未提供可核验的资料。"
)


def generate_answer(question, hits=None):
    user_message = f"【问题】{question}"
    if hits is not None:
        context = "\n\n".join(
            f"[{i}]（{source_label(doc)}）\n{doc.page_content}"
            for i, doc in enumerate(hits, 1)
        )
        user_message = f"【资料】\n{context}\n\n{user_message}"
    response = llm.invoke([("system", SYSTEM_PROMPT), ("user", user_message)])
    return response.content.strip()


baseline_answers = {}
if RUN_LIVE:
    for question in blind_questions:
        answer = generate_answer(question)
        baseline_answers[question] = answer
        print(f"【问】{question}\n【无资料】{answer}\n")
else:
    print("离线模式：跳过无资料基线。")

离线模式：跳过无资料基线。


读无资料基线时，把「内容是否正确」和「能否核验」分开看。即使答对最高峰，也要回到原文核对。对于万年寺，比较模型有没有使用讲解词里的称呼，并把砖殿内部的陈设与邻近行愿楼供奉的文物分开。

## 2. 文档加载：粒度决定后面怎么切

`Document.page_content` 保存正文，`metadata` 可以保存文件名、页码、标题路径、段落位置等信息。

| 粒度 | 一个 Document 里有什么 | 如何使用 |
|---|---|---|
| 整篇 | 整个文件 | 方便做纯文本基线，长文一般要继续切分 |
| 按页 | PDF 的一页 | 便于按页追溯，段落可能跨页 |
| 按元素 | 一个段落、标题、列表项或表格 | 可按结构聚合，再形成检索块 |
| 按块 | 实际参与检索的一组内容 | 块长和内容边界由切分策略决定 |

### 2.1 最省事的加载：整篇一个 Document

`Docx2txtLoader` 抽取 DOCX 文本，作为整篇一个 Document 的示例。

In [3]:
from langchain_community.document_loaders import Docx2txtLoader

t0 = time.perf_counter()
whole = Docx2txtLoader(str(DOCX)).load()

print(f"Document 数量：{len(whole)}")
print(f"总字符数    ：{sum(len(d.page_content) for d in whole)}")
print(f"耗时        ：{time.perf_counter() - t0:.2f}s")
print(f"metadata    ：{whole[0].metadata}")
print(f"开头 120 字 ：{whole[0].page_content[:120]!r}")
print("\n👉 整篇入库 = 1 个 chunk。用户问「第（十四）个景点」，检索只能选中整篇，后续还需要定位对应段落。")

Document 数量：1
总字符数    ：73900
耗时        ：0.05s
metadata    ：{'source': 'D:\\project\\step1\\week25\\20261109RAG\\峨眉山讲解词.docx'}
开头 120 字 ：'峨眉山导游词\n\n峨眉山概况\n\n峨眉山，雄踞四川盆地西南边缘，素有“仙山佛国”、“植物王国”、“动物乐园”和“地质博物馆”、“全国武术之乡”之美誉。其核心景区面积达154平方公里，最高峰万佛顶海拔3099米，主峰金顶3079米，以 “秀甲天下'

👉 整篇入库 = 1 个 chunk。用户问「第（十四）个景点」，检索只能选中整篇，后续还需要定位对应段落。


### 2.2 段落级：`python-docx` + 样式

要保住语义边界，就得逐个段落拿。`python-docx` 最直接：`Document.paragraphs` 是段落列表，每段带 `style`。

课堂用 `UnstructuredLoader` 是同一个思路的"加强版"（它还认表格、列表、页眉页脚）。先看看这份文档的样式有多可靠。

In [4]:
import docx

d = docx.Document(str(DOCX))
paras = [(p.text.strip(), p.style.name if p.style is not None else "") for p in d.paragraphs]
nonempty = [(t, s) for t, s in paras if t]

style_count = {}
for _, s in nonempty:
    style_count[s] = style_count.get(s, 0) + 1

print(f"段落总数：{len(paras)}    非空段：{len(nonempty)}")
print(f"样式分布：{style_count}")
print("\n抽 3 个看着像标题的段落，看它们的真实样式：")
for t, s in nonempty:
    if t.startswith(("（十四）", "四、", "1.恭敬")):
        print(f"  style={s!r:20s} text={t[:40]!r}")

段落总数：436    非空段：432
样式分布：{'Heading 1': 1, 'List Paragraph': 1, 'Normal': 429, 'Body Text 2': 1}

抽 3 个看着像标题的段落，看它们的真实样式：
  style='Normal'             text='1.恭敬佛宝'
  style='Normal'             text='四、峨眉山景点解说'
  style='Normal'             text='（十四）万年寺：无梁砖殿，第一山碑'


这份文件的 432 个非空段落中，`Heading 1` 只有 1 个，`Normal` 有 429 个。很多标题实际靠 `一、`、`（一）`、`1.` 区分层级，可以用文本模式补上章节路径。

### 2.3 元素级：课堂加载器与本机实现

课堂代码引用的是 `langchain_unstructured.UnstructuredLoader`。旧的 `UnstructuredFileLoader` 支持 `mode="single"/"elements"/"paged"`。新版 `UnstructuredLoader` 产出文档元素，可用 `chunking_strategy` 聚合元素。课堂的 `mode="elements"` 适合用来理解输出粒度，复跑时应按实际安装的接口写法。 [LangChain 集成文档](https://github.com/langchain-ai/docs/blob/main/src/oss/python/integrations/document_loaders/unstructured_file.mdx#chunking)

```python
# 装好对应依赖后，可以对照输出。本讲主流程使用下面的本机实现。
from langchain_unstructured import UnstructuredLoader

docs = UnstructuredLoader(str(DOCX)).load()
skip_categories = {"Header", "Footer", "PageNumber"}
filtered = [doc for doc in docs if doc.metadata.get("category") not in skip_categories]
```

下面用 `python-docx` 读取正文段落，按编号补出标题路径。此处的规则针对这份讲解词。处理复杂 PDF 时，可在这里接入 MinerU 一类的解析器，从结构化输出提取文本、表格和页码，再交给后面的清洗与切块。

In [5]:
HEAD_PATTERNS = [
    (1, re.compile(r"^[一二三四五六七八九十]+、")),      # 一、二、
    (2, re.compile(r"^（[一二三四五六七八九十]+）")),     # （一）（二）
    (3, re.compile(r"^\d+[\.、]")),                        # 1.  2、
]

def heading_level(text: str) -> int:
    """只把短文本当标题，避免把正文里以"1."开头的长句误判。"""
    if len(text) > 40:
        return 0
    for level, pat in HEAD_PATTERNS:
        if pat.match(text):
            return level
    return 0

def load_docx_elements(path: Path) -> list:
    """把 docx 拆成元素级 Document，metadata 记录 category / level / 小节路径。"""
    from langchain_core.documents import Document

    elements, path_stack, section_id = [], [], -1
    for paragraph_no, p in enumerate(docx.Document(str(path)).paragraphs, 1):
        text = p.text.strip()
        if not text:
            continue
        level = heading_level(text)
        if level:
            path_stack = path_stack[: level - 1] + [text]
            section_id += 1
            category = "Title" if level == 1 else "Heading"
        else:
            category = "Text"
        elements.append(Document(
            page_content=text,
            metadata={
                "source": path.name,
                "paragraph_no": paragraph_no,
                "category": category,
                "level": level,
                "section_id": section_id,
                "section": path_stack[-1] if path_stack else "（前言）",
                "path": " > ".join(path_stack) if path_stack else "（前言）",
            },
        ))
    return elements

elements = load_docx_elements(DOCX)
cat_count, lvl_count = {}, {}
for e in elements:
    cat_count[e.metadata["category"]] = cat_count.get(e.metadata["category"], 0) + 1
    lvl_count[e.metadata["level"]] = lvl_count.get(e.metadata["level"], 0) + 1

print(f"元素总数：{len(elements)}   category 分布：{cat_count}")
print(f"标题级别分布（0=正文）：{lvl_count}")
print("\n查看几条带路径的元素：")
for e in elements[:2] + [x for x in elements if x.metadata["level"] == 3][:2]:
    print(f"  {e.metadata['category']:8s} L{e.metadata['level']}  "
          f"{e.metadata['path'][:46]:46s} | {e.page_content[:34]}")

元素总数：432   category 分布：{'Text': 387, 'Title': 5, 'Heading': 40}
标题级别分布（0=正文）：{0: 387, 1: 5, 2: 37, 3: 3}

查看几条带路径的元素：
  Text     L0  （前言）                                           | 峨眉山导游词
  Text     L0  （前言）                                           | 峨眉山概况
  Heading  L3  二、佛教礼仪常识 > （二）礼敬佛、法、僧三宝 > 1.恭敬佛宝               | 1.恭敬佛宝
  Heading  L3  二、佛教礼仪常识 > （二）礼敬佛、法、僧三宝 > 2.恭敬法宝               | 2.恭敬法宝


### 2.4 按页加载：`PyMuPDFLoader`

课堂用 `car_info.pdf` 演示 PDF 加载。本机示例临时生成一个两页 PDF，并显式指定 `mode="page"`。每一页成为一个 `Document`。示例里的 `page` 从 0 开始，面向读者引用时可以显示为第 1、2 页。

In [6]:
import pymupdf
from langchain_community.document_loaders import PyMuPDFLoader

src = pymupdf.open()
page1 = src.new_page()
page1.insert_text((72, 100), "Page 1: Emei Mountain guide", fontsize=14)
page1.insert_text((72, 130), "Highest peak: Wanfoding, 3099 m", fontsize=11)
page2 = src.new_page()
page2.insert_text((72, 100), "Page 2: Buddhist etiquette", fontsize=14)
with TemporaryDirectory(prefix="rag_pdf_demo_") as tmp:
    pdf_demo = Path(tmp) / "two_pages.pdf"
    src.save(str(pdf_demo))
    src.close()
    pdf_docs = PyMuPDFLoader(str(pdf_demo), mode="page").load()
print(f"页数 = Document 数 = {len(pdf_docs)}")
for doc in pdf_docs:
    print(f"  page={doc.metadata.get('page')}  text={doc.page_content.strip()[:46]!r}")
print(f"\nmetadata 全量（第 1 页）：{dict(list(pdf_docs[0].metadata.items())[:8])}")
print("\n👉 按页加载的粒度由 PDF 排版决定：一页可能只有 3 行，也可能是 30 行长文。")

页数 = Document 数 = 2
  page=0  text='Page 1: Emei Mountain guide\nHighest peak: Wanf'
  page=1  text='Page 2: Buddhist etiquette'

metadata 全量（第 1 页）：{'producer': '', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\13744\\AppData\\Local\\Packages\\sandbox.{573f1ac0-9296-4e7c-8f76-bcf2485b9a2b}\\AC\\Temp\\rag_pdf_demo_ah5qkesw\\two_pages.pdf', 'file_path': 'C:\\Users\\13744\\AppData\\Local\\Packages\\sandbox.{573f1ac0-9296-4e7c-8f76-bcf2485b9a2b}\\AC\\Temp\\rag_pdf_demo_ah5qkesw\\two_pages.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': ''}

👉 按页加载的粒度由 PDF 排版决定：一页可能只有 3 行，也可能是 30 行长文。


## 3. 数据清洗：先检查，再处理排版

典型问题是 PDF 抽取后的硬换行，比如 `万佛\n顶`。处理前先统计换行、制表符、重复页眉等噪声，再按文档类型选规则。把断开的词修回去，尽量保留标题、专名、数字和单位。

本讲在逐段清洗后再切块，便于修复同一段内部的折行。跨段和跨页的问题需要结合解析器保留的版面结构。

### 3.1 自写正则：修折行 + 修汉字间空白

课堂的函数只处理 `\n`；这里再加一条"汉字之间的空白"（这份 docx 里只有 2 处，PDF 里会多得多）。

验证分两步：先拿一段**刻意构造的脏文本**（汉字间折行、标题词粘连、连续空行、URL/邮箱/电话）证明规则确实有效，再对真实全文做一遍统计，看它到底需不需要洗。

In [7]:
SECTION_HEADERS = "提示信息|警告！|注意！|说明！|帮助信息|图标和插图说明"

def clean_text_cn(text: str) -> str:
    """在课堂 clean_pdf_text 的基础上，增加「汉字之间的空白」清除。"""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\u3000]+", " ", text)                                  # 全角/多空格归一
    text = re.sub(r"(?<=[\u4e00-\u9fff])[ \t\u3000]+(?=[\u4e00-\u9fff])", "", text)  # 汉字间不留空格
    text = re.sub(r"(?<=[\u4e00-\u9fff])\n(?=[\u4e00-\u9fff])", "", text)      # 汉字间不留折行
    text = re.sub(r"(?<=[，。；：！？、）」』]) +", "", text)                    # 标点后不留空格
    text = re.sub(rf"({SECTION_HEADERS})(?=[\u4e00-\u9fff])", r"\1\n", text)   # 标题词后恢复换行
    text = re.sub(r"\n{3,}", "\n\n", text)                                    # 压缩空行
    return text.strip()

# ① 用一个「PDF 风格」的脏样本验证规则（这份 docx 里没有这种噪声，PDF 里到处都是）
dirty = (
    "峨眉山最高峰万佛\n顶海拔3099米，主峰金顶3079\n米。\n\n\n\n"
    "提示信息警告！请勿在寺院内大声喧哗 。\n"
    "官网 https://www.ems517.com ，邮箱 guide@emei.cn ，电话 0833-5520451。"
)
print("脏样本：", repr(dirty))
print("清洗后：", repr(clean_text_cn(dirty)))

# ② 真实文档：只统计，不做激进清洗
def inner_space_count(texts):
    pat = r"(?<=[\u4e00-\u9fff])[ \t\u3000]+(?=[\u4e00-\u9fff])"
    return sum(len(re.findall(pat, t)) for t in texts)

para_texts = [e.page_content for e in elements]
n_tab = sum(t.count("\t") for t in para_texts)
n_blank = sum(len(re.findall(r"\n{2,}", t)) for t in para_texts)

print(f"\n真实文档噪声统计（{len(para_texts)} 段）：")
print(f"  汉字之间的空格/全角空格：{inner_space_count(para_texts)} 处")
print(f"  制表符：{n_tab} 个")
print(f"  单段内的连续空行：{n_blank} 处")

raw_all = "\n".join(para_texts)
clean_all = "\n".join(clean_text_cn(t) for t in para_texts)
print(f"  全文长度：{len(raw_all)} → {len(clean_all)}（差 {len(raw_all) - len(clean_all)} 字符）")
print("\n👉 结论：这份 DOCX 的字符数变化很小。换成 PDF 时，先统计真实噪声，再决定用哪些规则。")
print("   「先统计、再清洗」比「先清洗、再祈祷」靠谱。")

脏样本： '峨眉山最高峰万佛\n顶海拔3099米，主峰金顶3079\n米。\n\n\n\n提示信息警告！请勿在寺院内大声喧哗 。\n官网 https://www.ems517.com ，邮箱 guide@emei.cn ，电话 0833-5520451。'
清洗后： '峨眉山最高峰万佛顶海拔3099米，主峰金顶3079\n米。\n\n提示信息\n警告！\n请勿在寺院内大声喧哗 。\n官网 https://www.ems517.com ，邮箱 guide@emei.cn ，电话 0833-5520451。'

真实文档噪声统计（432 段）：
  汉字之间的空格/全角空格：2 处
  制表符：2 个
  单段内的连续空行：0 处
  全文长度：73398 → 73378（差 20 字符）

👉 结论：这份 DOCX 的字符数变化很小。换成 PDF 时，先统计真实噪声，再决定用哪些规则。
   「先统计、再清洗」比「先清洗、再祈祷」靠谱。


输出里藏了三条**边界**，正好说明"清洗规则永远是局部解"：

1. `万佛\n顶` 被合上了，但 `3079\n米` **没有** —— 因为规则只写了「汉字-汉字」；"数字-汉字"要另写一条 `(?<=[0-9])\n(?=[\u4e00-\u9fff])`。
2. `喧哗 。` 里的空格还在 —— 这条规则治的是"标点**后**"，不治"标点**前**"。
3. URL / 邮箱 / 电话原封不动 —— 本例的规则只修排版，所以保留了联系方式。

所以工业上不会只写一条大正则，而是**一串小规则 + 一个"改了什么"的日志**：每加一条规则，都要能量出它在这份文档上命中了多少次（就像上面 `inner_space_count` 那样）。没有计数的清洗规则，等于没有测试的重构。

### 3.2 `clean-text`：分别看每个开关的影响

在本讲环境里，`no_numbers=True` 把 `3099` 替换成 `<number>`，`no_digits=True` 则把它变成 `0000`。讲解词的海拔、面积和年代经常就是答案，因此入库时应保留这些值。下面也能观察到默认清洗对全角标点的影响。

In [8]:
from cleantext import clean

demo = "联系 guide@emei.cn 或访问 https://www.ems517.com ，咨询电话 0833-5520451。"

print("原始    ：", demo)
print("去联系方式：", clean(demo, no_urls=True, no_emails=True, no_phone_numbers=True, to_ascii=False))

fact = "最高峰万佛顶海拔3099米，核心景区面积154平方公里。"
print("\n原始        ：", fact)
print("保留数字    ：", clean(fact, to_ascii=False))
print("no_numbers  ：", clean(fact, no_numbers=True, to_ascii=False))
print("no_digits   ：", clean(fact, no_digits=True, to_ascii=False))
print("\n👉 清洗规则一定要对着「你要回答的问题类型」来定：讲解词的核心信息就是数字和专名。")

原始    ： 联系 guide@emei.cn 或访问 https://www.ems517.com ，咨询电话 0833-5520451。
去联系方式： 联系 <email> 或访问 <url> ,咨询电话 <phone>。

原始        ： 最高峰万佛顶海拔3099米，核心景区面积154平方公里。
保留数字    ： 最高峰万佛顶海拔3099米,核心景区面积154平方公里。
no_numbers  ： 最高峰万佛顶海拔3099米,核心景区面积154平方公里。
no_digits   ： 最高峰万佛顶海拔0000米,核心景区面积000平方公里。

👉 清洗规则一定要对着「你要回答的问题类型」来定：讲解词的核心信息就是数字和专名。


Since the GPL-licensed package `unidecode` is not installed, using Python's `unicodedata` package which yields worse results.


### 3.3 课堂的 jionlp 示例

课堂使用 `clean_text()` 清理文本，使用 `split_sentence()` 识别中文句界。本讲使用可检查的正则规则演示这两步。比较方案时可以选几条带小数、单位、引号和换行的句子，核对它们有没有破坏原意。

```python
# 课堂接口，按需安装后运行。
import jionlp as jio

jio.clean_text("请拨打电话13800138000")
jio.split_sentence("万佛顶海拔3099米。金顶有哪些奇观？")
```

## 4. 文本分割：块长、边界和重叠

`chunk_size` 指定块长预算，单位由分割器的长度函数决定。下面的递归字符分割器默认用字符数，Token 分割器用分词后的 token 数。`chunk_overlap` 指定重叠预算。按段落或句号合并时，真实重叠还取决于边界。

| 分割器 | 适用场景 |
|---|---|
| `RecursiveCharacterTextSplitter` | 按给定的分隔符优先级递归切长文本 |
| `TokenTextSplitter` | 按指定 tokenizer 的 token 数切分 |
| `MarkdownHeaderTextSplitter` | 按 Markdown 标题组织正文和层级信息 |

### 4.1 递归切块，以及一个容易被忽略的坑

先看最常规的用法，同时对照两种"喂法"：

- `split_text(整篇字符串)` → 可以在整篇范围内自由合并，块数少；
- `split_documents([逐段 Document])` → **每个 Document 单独切，绝不跨 Document 合并**。

后者是课堂 `RAG2.ipynb` 和 `test3.py` 的写法（`text_splitter.split_documents(docs)`）：它之所以块数合理，是因为 `docs` 是"每页一个 Document"，一页里本来就包含很多段。**如果上游换成"每段一个 Document"（元素级加载），同一行代码就会产出几百个碎块。** 这就是"加载粒度"和"切块结果"的耦合。

In [9]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

clean_elements = [
    Document(page_content=clean_text_cn(e.page_content), metadata=dict(e.metadata))
    for e in elements
]
clean_elements = [d for d in clean_elements if d.page_content]

window_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", ""],
)

joined = "\n".join(d.page_content for d in clean_elements)
chunks_by_text = window_splitter.split_text(joined)
chunks_by_docs = window_splitter.split_documents(clean_elements)

def describe(chunks, name):
    lens = [len(c) if isinstance(c, str) else len(c.page_content) for c in chunks]
    print(f"{name:26s} 块数={len(chunks):4d}  平均={sum(lens)//len(lens):4d}  "
          f"最短={min(lens):3d}  最长={max(lens):4d}")

describe(chunks_by_text, "split_text(整篇)")
describe(chunks_by_docs, "split_documents(逐段)")

# 受控实验：10 段拼成一条无换行、约 3600 字的长文本，只改「有没有句号分隔符」
long_para = "".join(d.page_content for d in clean_elements[2:12])
print(f"\n受控实验长文本：{len(long_para)} 字（无换行，chunk_size=300）")

by_sentence = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30, separators=["。", "！", "？", "；", ""]
).split_text(long_para)
by_char = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30, separators=[""]
).split_text(long_para)

print(f"  按句号切  ：{len(by_sentence)} 块，块长 {[len(c) for c in by_sentence][:8]} …")
print(f"  按单字硬切：{len(by_char)} 块，块长 {[len(c) for c in by_char][:8]} …")

print("\n边界长什么样：")
for i in (0, 1):
    print(f"  句号切 {i}  尾 {by_sentence[i][-16:]!r} | 头 {by_sentence[i + 1][:16]!r}")
    print(f"  硬切   {i}  尾 {by_char[i][-16:]!r} | 头 {by_char[i + 1][:16]!r}")


split_text(整篇)             块数= 201  平均= 377  最短= 66  最长= 499
split_documents(逐段)        块数= 449  平均= 164  最短=  1  最长= 500

受控实验长文本：3645 字（无换行，chunk_size=300）
  按句号切  ：14 块，块长 [272, 255, 270, 227, 287, 285, 279, 233] …
  按单字硬切：14 块，块长 [300, 300, 300, 300, 300, 300, 300, 300] …

边界长什么样：
  句号切 0  尾 '豪郭沫若题书峨眉山为“天下名山”' | 头 '。古往今来，峨眉山就是人们礼佛朝'
  硬切   0  尾 '们礼佛朝拜、游览观光、科学考察和' | 头 '山”。古往今来，峨眉山就是人们礼'
  句号切 1  尾 '之峨，山秀谓之眉，故谓之“峨眉”' | 头 '。峨眉山名，早见于晋代常璩撰写的'
  硬切   1  尾 '：“杜宇以褒斜（今陕西汉中）为前' | 头 '撰写的《华阳国志•蜀志》记载：“'


同一份讲解词用 `chunk_size=500` 递归切分，整篇输入得到 **201 块、平均 377 字**，逐段输入得到 **449 块、平均 164 字**。因为 `split_documents()` 对每个输入 `Document` 分别处理，短段落也会成为独立块。需要完整小节时，可以先聚合，再调用切块器。

受控例子用句末标点作为优先边界，超过预算的句子仍可能退回按单字切分。默认保留分隔符时，句号可能留在下一块开头。想把句号留在上一块，可以显式指定 `keep_separator="end"`，并用原文检查边界。

### 4.2 正则分隔符与 `is_separator_regex`

课堂分隔符 `(?<=\。！？ )` 有两处需要区分。关闭 `is_separator_regex` 时，分割器按字面字符串匹配。打开后，这个后顾表达式仍要求前面连续出现字面序列 `。！？ `，其中包括末尾空格。用 `(?<=[。！？])` 才会在任意一个中文句末标点后匹配。

下面在同一段文本里比较三种配置。

In [10]:
sample = (
    "峨眉山位于四川盆地西南边缘。最高峰万佛顶海拔3099米！主峰金顶3079米。"
    "金顶有日出、云海、佛光、晚霞四大奇观。景区核心面积154平方公里。"
)

def cut(seps, is_regex, size=40):
    sp = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=0,
        separators=seps, is_separator_regex=is_regex,
    )
    return sp.split_text(sample)

cases = [
    ("A 课堂写法（正则当普通串）", ["\n\n", "\n", r"(?<=\。！？ )", " ", ""], False),
    ("B 开正则，但要求连续三个标点和空格", ["\n\n", "\n", r"(?<=\。！？ )", " ", ""], True),
    ("C 正确：不用空格+开正则", ["\n\n", "\n", "(?<=[。！？])", " ", ""], True),
]
for label, seps, is_re in cases:
    out = cut(seps, is_re)
    print(f"{label} → {len(out)} 块，各自长度 {[len(x) for x in out]}")
    for piece in out:
        print(f"      {piece!r}")
    print()

A 课堂写法（正则当普通串） → 2 块，各自长度 [40, 31]
      '峨眉山位于四川盆地西南边缘。最高峰万佛顶海拔3099米！主峰金顶3079米。金顶'
      '有日出、云海、佛光、晚霞四大奇观。景区核心面积154平方公里。'

B 开正则，但要求连续三个标点和空格 → 2 块，各自长度 [40, 31]
      '峨眉山位于四川盆地西南边缘。最高峰万佛顶海拔3099米！主峰金顶3079米。金顶'
      '有日出、云海、佛光、晚霞四大奇观。景区核心面积154平方公里。'

C 正确：不用空格+开正则 → 2 块，各自长度 [38, 33]
      '峨眉山位于四川盆地西南边缘。最高峰万佛顶海拔3099米！主峰金顶3079米。'
      '金顶有日出、云海、佛光、晚霞四大奇观。景区核心面积154平方公里。'



本例 A 和 B 没有命中句末边界，回退到字符切分，第一块达到 40 字。C 在句号后断开，第一块是 38 字。排查时可以先用 `re.finditer()` 看分隔规则是否命中，再确认分割器开启了正则模式。

### 4.3 `chunk_overlap` 到底重叠了什么

在下面的字符窗口中，配置 60 字重叠得到 `[60, 60, 60]`。优先按句号切时，同样配置得到 `[0, 0, 60]`。读检索结果时，可以检查关键信息是否完整落在某一块里。跨章节或跨输入文档的上下文要由上游聚合来处理。

In [11]:
plain = "".join(d.page_content for d in clean_elements[2:8])

def tail_head_overlap(a: str, b: str) -> int:
    """a 的后缀与 b 的前缀最长重合长度。"""
    for n in range(min(len(a), len(b)), 0, -1):
        if a.endswith(b[:n]):
            return n
    return 0

for label, seps in [("纯字符窗口 separators=['']", [""]), ("按句号切 separators=['。','']", ["。", ""])]:
    sp = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=60, separators=seps)
    ch = sp.split_text(plain)[:4]
    overlaps = [tail_head_overlap(ch[i], ch[i + 1]) for i in range(len(ch) - 1)]
    print(f"{label}")
    print(f"  块长 {[len(c) for c in ch]}   实际重叠 {overlaps}")
    for i, k in enumerate(overlaps):
        if k:
            print(f"    块{i} 末 {k} 字  {ch[i][-k:]!r}")
            print(f"    块{i + 1} 首 {k} 字  {ch[i + 1][:k]!r}   ← 与上一行完全相同")
    print()


纯字符窗口 separators=['']
  块长 [180, 180, 180, 180]   实际重叠 [60, 60, 60]
    块0 末 60 字  '，厚重的佛教文化和独具特色的人文景观享誉中外，被联合国专家称之为世界上同纬度生物、植物保护最好的区域，更是以其“雄、秀、'
    块1 首 60 字  '，厚重的佛教文化和独具特色的人文景观享誉中外，被联合国专家称之为世界上同纬度生物、植物保护最好的区域，更是以其“雄、秀、'   ← 与上一行完全相同
    块1 末 60 字  '甲天下，何须涉海寻蓬莱”，当代文豪郭沫若题书峨眉山为“天下名山”。古往今来，峨眉山就是人们礼佛朝拜、游览观光、科学考察和'
    块2 首 60 字  '甲天下，何须涉海寻蓬莱”，当代文豪郭沫若题书峨眉山为“天下名山”。古往今来，峨眉山就是人们礼佛朝拜、游览观光、科学考察和'   ← 与上一行完全相同
    块2 末 60 字  '入。古人为山命名，通常是根据形态和特点，峨眉山因秀似“蛾眉”而得名。我国地理学奠基人之一的郦道元在《水经注》中记载：“去'
    块3 首 60 字  '入。古人为山命名，通常是根据形态和特点，峨眉山因秀似“蛾眉”而得名。我国地理学奠基人之一的郦道元在《水经注》中记载：“去'   ← 与上一行完全相同

按句号切 separators=['。','']
  块长 [57, 167, 169, 155]   实际重叠 [0, 0, 60]
    块2 末 60 字  '。起初这里是道家的一个清修之地，公元一世纪时佛教开始传入。古人为山命名，通常是根据形态和特点，峨眉山因秀似“蛾眉”而得名'
    块3 首 60 字  '。起初这里是道家的一个清修之地，公元一世纪时佛教开始传入。古人为山命名，通常是根据形态和特点，峨眉山因秀似“蛾眉”而得名'   ← 与上一行完全相同



### 4.4 按 Token 切和按标题切

- Token 数取决于 tokenizer 和原文。下面用 `cl100k_base` 计算，样例的 47 个字符约为 66 token。首次使用可能下载分词文件，此演示随云端模式开启。
- `MarkdownHeaderTextSplitter` 识别 `#`、`##`、`###`，把标题层级写入元数据。下一个示例用相同的思路处理 Word 文档里的编号标题。

In [12]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

if RUN_LIVE:
    import tiktoken
    from langchain_text_splitters import TokenTextSplitter

    enc = tiktoken.get_encoding("cl100k_base")
    probe = "峨眉山，雄踞四川盆地西南边缘，素有“仙山佛国”“植物王国”等美誉。最高峰万佛顶海拔3099米。"
    print(f"tiktoken 计数：{len(probe)} 字符 → {len(enc.encode(probe))} token"
          f"（{len(enc.encode(probe)) / len(probe):.2f} token/字符）")

    token_splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=60, chunk_overlap=10)
    tok_chunks = token_splitter.split_text(probe)
    print(f"TokenTextSplitter(60/10) → {len(tok_chunks)} 块，字符长度 {[len(c) for c in tok_chunks]}")
else:
    print("离线模式：跳过可能需要下载分词文件的 Token 示例。")

md_text = """# 峨眉山导游词

## 二、佛教礼仪常识

### （一）礼仪
礼仪是指人们在社会交往活动中形成的行为规范与准则。

### （二）礼敬佛、法、僧三宝
合十、问讯、礼拜是佛教信众日常生活中最常用的礼节。
"""
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])
for i, piece in enumerate(md_splitter.split_text(md_text)):
    print(f"块{i}: metadata={piece.metadata}  → {piece.page_content[:24]!r}")

离线模式：跳过可能需要下载分词文件的 Token 示例。
块0: metadata={'h1': '峨眉山导游词', 'h2': '二、佛教礼仪常识', 'h3': '（一）礼仪'}  → '礼仪是指人们在社会交往活动中形成的行为规范与准则'
块1: metadata={'h1': '峨眉山导游词', 'h2': '二、佛教礼仪常识', 'h3': '（二）礼敬佛、法、僧三宝'}  → '合十、问讯、礼拜是佛教信众日常生活中最常用的礼节'


### 4.5 按标题聚合，再对长节切块

按已识别的标题层级把连续段落聚成节。短节可以和相邻段落合并，超过预算的正文再交给递归分割器。合并后的块记录**首尾章节路径和段落范围**，跨节引用时显示范围。拆分长节后的段落范围仍对应原聚合段，`part` 用来区分该段下的各块。

本节通过输出读取块长和数量。块数较少只反映切分策略不同，第 10 节用同一批问题比较两种切法的证据覆盖。父子块的「小块检索、回填父块」在待办里单独列出。

In [13]:
from itertools import groupby


def source_label(doc):
    metadata = doc.metadata
    first = metadata.get("path")
    last = metadata.get("path_last", first)
    chapter = f" · {first}" if first else ""
    if last != first:
        chapter += f" 至 {last}"
    if "para_start" in metadata:
        first_para, last_para = metadata["para_start"], metadata["para_end"]
        chapter += f" · 第 {first_para}–{last_para} 段"
    return metadata["source"] + chapter


def merge_sections(left, right):
    return {
        **left,
        "path_last": right["path_last"],
        "para_end": right["para_end"],
        "body": left["body"] + "\n" + right["body"],
    }


def build_structural_chunks(elements, target=1000, maxc=1200, overlap=120, min_section=120):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=maxc, chunk_overlap=overlap,
        separators=["\n\n", "\n", "。", "！", "？", "；", "，", ""],
    )
    sections = []
    for _, group in groupby(elements, key=lambda doc: doc.metadata["section_id"]):
        group = list(group)
        first, last = group[0].metadata, group[-1].metadata
        sections.append({
            "source": first["source"],
            "path": first["path"],
            "path_last": last["path"],
            "para_start": first["paragraph_no"],
            "para_end": last["paragraph_no"],
            "body": "\n".join(doc.page_content for doc in group),
        })

    kept, carry = [], None
    for section in sections:
        section = merge_sections(carry, section) if carry is not None else section
        if len(section["body"]) < min_section:
            carry = section
        else:
            kept.append(section)
            carry = None
    if carry is not None:
        if kept:
            kept[-1] = merge_sections(kept[-1], carry)
        else:
            kept.append(carry)

    merged = []
    for section in kept:
        if merged and len(merged[-1]["body"]) + 1 + len(section["body"]) <= target:
            merged[-1] = merge_sections(merged[-1], section)
        else:
            merged.append(section)

    chunks = []
    for section in merged:
        body = section["body"]
        metadata = {key: value for key, value in section.items() if key != "body"}
        pieces = splitter.split_text(body) if len(body) > maxc else [body]
        for part, piece in enumerate(pieces):
            chunk_id = f"{metadata['source']}:{metadata['para_start']}:{part}"
            chunks.append(Document(
                id=chunk_id,
                page_content=piece,
                metadata={**metadata, "part": part, "chars": len(piece), "category": "Section"},
            ))
    return chunks


structural_chunks = build_structural_chunks(clean_elements)
fixed_chunks = [
    Document(id=f"fixed:{i}", page_content=text, metadata={"source": DOCX.name})
    for i, text in enumerate(chunks_by_text)
]
describe(structural_chunks, "按标题聚合")
describe(fixed_chunks, "整篇递归切块")
print("\n前四块的出处范围：")
for chunk in structural_chunks[:4]:
    print(f"  [{len(chunk.page_content):4d} 字] {source_label(chunk)}")

按标题聚合                      块数=  87  平均= 851  最短= 72  最长=1200
整篇递归切块                     块数= 201  平均= 377  最短= 66  最长= 499

前四块的出处范围：
  [ 743 字] 峨眉山讲解词.docx · （前言） · 第 1–8 段
  [ 501 字] 峨眉山讲解词.docx · （前言） · 第 1–8 段
  [1180 字] 峨眉山讲解词.docx · （前言） · 第 1–8 段
  [ 211 字] 峨眉山讲解词.docx · （前言） · 第 1–8 段


## 5. 文本嵌入：在同一个空间里比较文本

嵌入模型把文本映射成固定维度的向量，训练目标是让语义相关的内容更容易互相检索。相似度用来排序候选片段，能否回答还要检查正文。

本讲指定 `text-embedding-v4` 输出 1024 维。下面观察实际返回的范数。对于单位向量，余弦相似度等于点积。建库和查询需要使用同一套模型与对应的编码配置，向量长度相同也不能保证属于同一个空间。换嵌入模型或维度后应重算索引。 [阿里云百炼向量接口](https://help.aliyun.com/en/model-studio/embedding-interfaces-compatible-with-openai)

In [14]:
if RUN_LIVE:
    text = "峨眉山最高峰海拔多少米"
    v = embedding_model.embed_query(text)
    same = embedding_model.embed_documents([text])[0]
    norm = np.linalg.norm(v)
    cosine = float(np.dot(v, same) / (norm * np.linalg.norm(same)))
    print(f"模型：{EMBED_MODEL}，维度：{len(v)}，查询向量范数：{norm:.6f}")
    print(f"同一句分别作为查询和文档时的余弦：{cosine:.6f}")
else:
    print("离线模式：跳过嵌入请求。")

离线模式：跳过嵌入请求。


In [15]:
def cos_sim(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

if RUN_LIVE:
    sentences = [
        "峨眉山最高峰是万佛顶，海拔3099米。",
        "金顶是峨眉山的主峰，海拔3079米。",
        "峨眉山的猴子会抢游客的食物。",
    ]
    vecs = embedding_model.embed_documents(sentences)
    print("句子两两余弦相似度：")
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            print(f"  {i}-{j}: {cos_sim(vecs[i], vecs[j]):.4f}   "
                  f"{sentences[i][:12]} | {sentences[j][:12]}")
    print("\n👉 比较三组分数，观察相关主题在这批样本里的排序。")
else:
    print("离线模式：跳过嵌入请求。")

离线模式：跳过嵌入请求。


### 5.1 课堂那段"国际新闻"检索实验

课程用 5 条新闻 + 1 个 query 手写余弦排序，也试了英文查询 `global conflicts`。观察中文和英文查询在这五条样本里的排序。

In [16]:
if RUN_LIVE:
    news = [
        "联合国就苏丹达尔富尔地区大规模暴力事件发出警告",
        "土耳其、芬兰、瑞典与北约代表将继续就瑞典“入约”问题进行谈判",
        "日本岐阜市陆上自卫队射击场内发生枪击事件 3人受伤",
        "国家游泳中心（水立方）：恢复游泳、嬉水乐园等水上项目运营",
        "我国首次在空间站开展舱外辐射生物学暴露实验",
    ]
    news_vecs = embedding_model.embed_documents(news)

    for query in ["国际争端", "global conflicts", "游泳馆恢复营业"]:
        qv = embedding_model.embed_query(query)
        scored = sorted(zip(news, [cos_sim(qv, nv) for nv in news_vecs]),
                        key=lambda t: -t[1])
        print(f"query = {query!r}")
        for text, score in scored:
            print(f"   {score:.4f}  {text[:32]}")
        print()
else:
    print("离线模式：跳过嵌入请求。")

离线模式：跳过嵌入请求。


可以观察三件事：换语言是否仍能召回中文主题，换问法后候选片段的排名会不会变，以及第一、第二名的分差。这里的五条新闻只用于演示排序。实际项目需要用标注样本判断一个阈值是否合适，以及查询改写是否提高证据召回率。

## 6. Chroma：原生接口、LangChain 封装和更新

一条记录通常包括 ID、向量、原文和元数据。向量用于近邻搜索，原文用于回答，元数据支持过滤和引用。下面对照原生接口和 LangChain 封装，再演示如何更新已有集合。

### 6.1 原生 Chroma：显式传入向量

课堂用 `HttpClient` 连接服务，本节用临时集合演示相同的 `add()`、`query()` 调用。建库与查询都显式传入千问向量。这样向量来自同一套嵌入配置，示例也不依赖 Chroma 的默认嵌入函数。

In [17]:
if RUN_LIVE:
    import chromadb

    client = chromadb.EphemeralClient()
    col = client.create_collection("native_demo", get_or_create=True, embedding_function=None)
    docs_demo = ["小明喜欢吃苹果", "王老师是一个好老师"]
    col.upsert(
        ids=["d1", "d2"],
        documents=docs_demo,
        embeddings=embedding_model.embed_documents(docs_demo),
        metadatas=[{"src": "demo"}, {"src": "demo"}],
    )
    result = col.query(
        query_embeddings=[embedding_model.embed_query("谁是老师")],
        n_results=2,
        include=["documents", "metadatas", "distances"],
    )
    for key in ("ids", "distances", "documents", "metadatas"):
        print(f"{key}: {result[key]}")
else:
    print("离线模式：跳过原生向量检索示例。")

离线模式：跳过原生向量检索示例。


### 6.2 LangChain 封装：`Chroma.from_documents`

封装接收带正文和元数据的 `Document`，负责嵌入与写入。这里显式选择 cosine 距离。每次完整建库使用独立目录，方便在同一内核里重跑，也能避免与旧集合混用。目录前缀包含模型和维度。

在同一个集合里，重复写入同一个 ID 会更新该记录。每次生成新 ID 则会增加记录。这里使用切块阶段生成的 ID。真实资料换版后，仍要清理旧版本中已经失效的 ID。

In [18]:
from langchain_chroma import Chroma

vectordb = None
if RUN_LIVE:
    DB_DIR.mkdir(parents=True, exist_ok=True)
    prefix = f"emei_{EMBED_MODEL.replace('-', '_')}_{EMBED_DIM}_"
    main_dir = Path(mkdtemp(prefix=prefix, dir=DB_DIR))
    t0 = time.perf_counter()
    vectordb = Chroma.from_documents(
        documents=structural_chunks,
        ids=[doc.id for doc in structural_chunks],
        embedding=embedding_model,
        collection_name="emei_guide",
        persist_directory=str(main_dir),
        collection_metadata={"hnsw:space": "cosine"},
    )
    count = len(vectordb.get(include=[])["ids"])
    if count != len(structural_chunks):
        raise RuntimeError("建库记录数与切块数不一致。")
    print(f"建库完成：{count} 条，耗时 {time.perf_counter() - t0:.1f}s")
    print(f"落盘目录：{main_dir.name}")
else:
    print("离线模式：跳过建库。")

离线模式：跳过建库。


### 6.3 增量更新：按 ID 写入和清理

重开上一节的集合，写入一条标记为虚构的门票演示片段。用相同 ID 再写一次，比较两次写入后的记录数。用 `finally` 清理这条演示数据，保证后面仍可以测试资料缺失的情形。查询结果按距离排序。业务里的资料生效日期、版本和来源需要由系统另外管理。

In [19]:
if RUN_LIVE:
    db = Chroma(
        collection_name="emei_guide",
        embedding_function=embedding_model,
        persist_directory=str(main_dir),
    )
    demo_id = "__lesson_ticket_demo__"
    demo_text = "【虚构练习数据】景区门票演示价为100元，仅供验证更新流程。"
    demo_doc = Document(
        id=demo_id,
        page_content=demo_text,
        metadata={"source": "虚构练习数据", "path": "门票演示", "chars": len(demo_text)},
    )
    before = len(db.get(include=[])["ids"])
    try:
        db.add_documents([demo_doc], ids=[demo_id])
        first_count = len(db.get(include=[])["ids"])
        db.add_documents([demo_doc], ids=[demo_id])
        repeat_count = len(db.get(include=[])["ids"])
        print(f"原集合 {before} 条，写入后 {first_count} 条，重写同 ID 后 {repeat_count} 条")
        hits = db.similarity_search_with_score("峨眉山门票多少钱？", k=3)
        for doc, distance in hits:
            print(f"距离={distance:.4f}  {source_label(doc)}")
    finally:
        db.delete(ids=[demo_id])
    print(f"清理演示记录后：{len(db.get(include=[])['ids'])} 条")
else:
    print("离线模式：跳过增量更新。")

离线模式：跳过增量更新。


## 7. 检索：从近邻结果找到答案证据

每次检索可以观察距离、前几名里有没有证据，以及现有证据能否答全问题。

### 7.1 读懂 Chroma 返回的距离

本讲用 cosine 集合。`similarity_search_with_score()` 返回的是距离，越小越接近。cosine 距离是 `1 - cos(a,b)`。对两个单位向量，平方 L2 距离是 `2 × (1 - cos(a,b))`。后者的换算以单位范数为前提。

下面用同一次查询和集合里已经存储的向量核对公式。

In [20]:
if RUN_LIVE:
    question = "峨眉山最高峰海拔多少米"
    qv = embedding_model.embed_query(question)
    raw = vectordb._collection.query(
        query_embeddings=[qv], n_results=3,
        include=["embeddings", "distances", "metadatas"],
    )
    for dv, distance, metadata in zip(raw["embeddings"][0], raw["distances"][0], raw["metadatas"][0]):
        cosine = cos_sim(qv, dv)
        l2_squared = np.sum((np.asarray(qv) - np.asarray(dv)) ** 2)
        print(f"Chroma 距离={distance:.4f}，1-cos={1-cosine:.4f}，平方 L2={l2_squared:.4f}")
        print(f"  章节：{metadata['path']}")
else:
    qv, dv = [1.0, 0.0], [0.8, 0.6]
    cosine = cos_sim(qv, dv)
    print(f"手算示例：1-cos={1-cosine:.2f}，平方 L2={np.sum((np.asarray(qv)-dv)**2):.2f}")

手算示例：1-cos=0.20，平方 L2=0.40


在离线示例里，两个向量的范数都为 1，cosine 距离是 0.20，平方 L2 是 0.40。线上例子以实际存储的向量和集合配置为准。

### 7.2 怎么选择 `k`

`k` 小，支持答案的片段可能排在结果之外。`k` 大，上下文会变长，也可能引入无关片段。先观察支持答案的片段排在第几位，再用固定问题集比较不同 `k` 的证据覆盖和上下文长度。

In [21]:
if RUN_LIVE:
    real_questions = [
        "峨眉山最高峰是哪座？海拔多少米？",
        "万年寺的砖殿为什么叫「无梁砖殿」？",
        "在寺院里应该遵守哪些礼仪？",
        "金顶有哪些奇观？",
    ]

    for q in real_questions:
        print(f"【问】{q}")
        for i, (doc, score) in enumerate(vectordb.similarity_search_with_score(q, k=3), 1):
            print(f"  {i}. 距离={score:.4f}  {source_label(doc)}")
            print(f"      {doc.page_content[:56].strip()!r}")
        print()
else:
    print("离线模式：本节需要 RUN_LIVE=True。")

离线模式：本节需要 RUN_LIVE=True。


### 7.3 区分缺证据和漏召回

| 问题 | 原文里有什么 | 怎么判断结果 |
|---|---|---|
| 门票多少钱、今天的天气如何 | 讲解词缺少这两个问题所需的信息 | 即使检索有返回，也要核对能否支持回答 |
| 第（十九）个景点是哪个 | 标题直接写了「（十九）洪椿坪」 | 如果 Top-k 没包含这个标题，是漏召回 |

近邻检索会按距离选候选片段。精确编号可以另外存入元数据，用编号过滤或关键词匹配定位。是否采用距离阈值，应以评测数据为依据。

In [22]:
if RUN_LIVE:
    scenarios = [
        ("资料缺失", "峨眉山门票多少钱？"),
        ("资料缺失", "今天峨眉山天气怎么样？"),
        ("原文有证据", "第（十九）个景点是哪个？"),
    ]
    for label, question in scenarios:
        print(f"【{label}】{question}")
        hits = vectordb.similarity_search_with_score(question, k=2)
        for doc, distance in hits:
            print(f"  距离={distance:.4f}  {source_label(doc)}")
        if label == "原文有证据":
            found = any("（十九）洪椿坪" in doc.page_content for doc, _ in hits)
            print(f"  Top-2 命中目标标题：{found}")
        print()
else:
    print("离线模式：跳过向量检索。第 10 节会校验编号题的原文证据。")

离线模式：跳过向量检索。第 10 节会校验编号题的原文证据。


## 8. 生成：让模型看到原文和出处

用第 1 节的同一模型、问题和系统提示，仅在用户消息里加入检索资料。每段资料带编号、来源和章节范围。模型以 `[1]` 这类编号引用资料，读者可以回查支持结论的原文。

下方打印的出处来自当次生成实际使用的 `hits`。第 7 节的近邻距离可以解释排序，是否有证据则由片段内容决定。

In [23]:
def answer_with_rag(question, k=4):
    hits = vectordb.similarity_search(question, k=k)
    return generate_answer(question, hits), hits


def print_evidence(hits, full=False):
    for i, doc in enumerate(hits, 1):
        text = doc.page_content if full else " ".join(doc.page_content.split())
        if not full and len(text) > 110:
            text = text[:110] + "…"
        print(f"  [{i}] {source_label(doc)}\n      {text}")


if RUN_LIVE:
    for question in blind_questions:
        answer, hits = answer_with_rag(question)
        print(f"【问】{question}\n【无资料】{baseline_answers[question]}\n【有资料】{answer}")
        print_evidence(hits)
        print()
else:
    print("离线模式：跳过云端问答。")

离线模式：跳过云端问答。


对照时逐项核对答案是否完整，以及引用的原文是否支持它。再用门票题检查缺证据时的行为。对于文档本来有答案但没有检索到的题，可以记录为检索遗漏，即使模型按当前上下文作出了合理的拒答。

In [24]:
if RUN_LIVE:
    question = "峨眉山的门票价格是多少？"
    answer, hits = answer_with_rag(question)
    print(f"【问】{question}\n【本次实际提供的资料】")
    print_evidence(hits, full=True)
    print(f"【答】{answer}")
else:
    print("离线模式：跳过门票题的生成。")

离线模式：跳过门票题的生成。


## 9. 工具化：把检索器交给 Agent

第 8 节由程序固定先检索再回答。这里把检索器包装成 `emei_guide_search`，让模型选择是否调用、如何写查询。对于景区事实，提示要求先调用工具。

检索工具的默认模板只返回正文。这里用 `document_prompt` 显式带上文件、章节和段落范围，再要求 Agent 在回答里附上工具提供的出处。运行后看工具参数和返回的第一行，核对最终回答有没有使用对应来源。

In [25]:
from langchain.agents import create_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools.retriever import create_retriever_tool

if RUN_LIVE:
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    retriever_tool = create_retriever_tool(
        retriever,
        name="emei_guide_search",
        description="检索《峨眉山讲解词》中的景点、海拔、佛教礼仪和游览线路。",
        document_prompt=PromptTemplate.from_template(
            "出处：{source} · 章节范围：{path} 至 {path_last} · 第 {para_start}–{para_end} 段\n"
            "{page_content}"
        ),
    )
    agent = create_agent(
        llm,
        tools=[retriever_tool],
        system_prompt=(
            "你是峨眉山景区讲解助手。景区事实先调用 emei_guide_search，"
            "依据工具返回的资料回答，并在结论旁注明对应文件和章节。"
            "资料不足时，回答「资料中没有相关信息」。"
        ),
    )
    result = agent.invoke({"messages": [
        {"role": "user", "content": "峨眉山最高峰叫什么？海拔多少米？"}
    ]})
    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print(f"工具：{call['name']}，参数：{call['args']}")
        elif message.type == "tool":
            print(f"工具返回：{str(message.content)[:250]}…")
    print(f"Agent 回答：{result['messages'][-1].content}")
else:
    print("离线模式：跳过 Agent 调用。")

离线模式：跳过 Agent 调用。


## 10. 小评测和课程回顾

先固定问题和原文证据，再比较切块策略。对照用同一嵌入模型、同一批查询向量、同样的 cosine 距离和 `k=4`。两种策略的块长不同，结果同时打印送入上下文的平均字符数。

### 10.1 用标注证据比较两种切法

每题标注参考答案和一组原文短语。Top-k 的正文包含所有锚点，记为证据全部命中。多证据题也记录覆盖比例。这个办法便于复跑，正式评测还要人工检查完整段落是否真的支持答案，特别是一个短语跨块或有多个等价出处时。

缺证据题单独看拒答。第 8 节负责检查答案内容和引用，本格的检索分数只衡量候选片段里的证据覆盖。

In [26]:
eval_cases = [
    ("最高峰是哪座，海拔是多少？", "万佛顶，3099 米。", ("最高峰万佛顶海拔3099米",)),
    ("主峰金顶多高？", "3079 米。", ("主峰金顶3079米",)),
    ("讲解词给出的核心景区面积是多少？", "154 平方公里。", ("核心景区面积达154平方公里",)),
    ("万年寺的砖殿为什么叫无梁砖殿？", "没有传统屋梁，殿壁和殿顶等由砖砌成。",
     ("砖殿因没有传统的屋梁", "门楣、额枋、斗拱、窗棂皆为砖砌")),
    ("砖殿上部供奉了多少尊小铁佛像？", "303 尊。", ("供奉303尊小铁佛像",)),
    ("万年寺三宝放在哪里，分别是什么？", "行愿楼，佛牙舍利、梵文贝叶经和御印。",
     ("行愿楼”，内供万年寺三宝", "佛牙舍利、梵文贝叶经和御印")),
    ("第（十九）个景点叫什么？", "洪椿坪。", ("（十九）洪椿坪",)),
    ("天池峰名字怎么来的？", "山上有天然石池。", ("因山上有天然石池而得名",)),
    ("金顶祥光由哪四种自然奇观组成？", "日出、云海、佛光、圣灯。",
     ("日出、云海、佛光、圣灯四大自然奇观",)),
    ("佛教信众常用的礼节有哪些？万年寺三宝在哪座楼？", "合十、问讯、礼拜。三宝在行愿楼。",
     ("合十、问讯、礼拜", "行愿楼”，内供万年寺三宝")),
    ("峨眉山的门票价格是多少？", None, ()),
    ("今天峨眉山景区实时气温是多少？", None, ()),
]

answerable = [(question, answer, anchors) for question, answer, anchors in eval_cases if anchors]
for question, _, anchors in answerable:
    missing = [anchor for anchor in anchors if anchor not in joined]
    if missing:
        raise ValueError(f"评测证据在清洗后原文中找不到：{question} / {missing}")
print(f"评测集：{len(eval_cases)} 题。已核对 {len(answerable)} 道有答案题的证据锚点。")


def evidence_coverage(hits, anchors):
    texts = [doc.page_content for doc in hits]
    return sum(any(anchor in text for text in texts) for anchor in anchors) / len(anchors)


if RUN_LIVE and RUN_EVAL:
    fixed_db = Chroma.from_documents(
        fixed_chunks,
        embedding_model,
        ids=[doc.id for doc in fixed_chunks],
        collection_name="emei_fixed_eval",
        client=chromadb.EphemeralClient(),
        collection_metadata={"hnsw:space": "cosine"},
    )
    k = 4
    queries = [embedding_model.embed_query(question) for question, _, _ in answerable]
    for label, store in (("整篇递归切块", fixed_db), ("按标题聚合", vectordb)):
        coverages, lengths = [], []
        print(f"\n【{label}】")
        for query, (question, _, anchors) in zip(queries, answerable):
            hits = store.similarity_search_by_vector(query, k=k)
            coverage = evidence_coverage(hits, anchors)
            coverages.append(coverage)
            lengths.append(sum(len(doc.page_content) for doc in hits))
            print(f"  证据覆盖 {coverage:.0%}  {question}")
        full_hits = sum(value == 1.0 for value in coverages)
        print(f"  Top-{k} 证据全部命中 {full_hits}/{len(answerable)}  "
              f"平均证据覆盖 {np.mean(coverages):.1%}  平均上下文 {np.mean(lengths):.0f} 字")

    print("\n【缺证据题】")
    for question, answer, anchors in eval_cases:
        if answer is None:
            generated, hits = answer_with_rag(question, k=k)
            print(f"{question}\n{generated}")
            print_evidence(hits)
else:
    print("打开 RUN_LIVE 和 RUN_EVAL 后运行对比。")

评测集：12 题。已核对 10 道有答案题的证据锚点。
打开 RUN_LIVE 和 RUN_EVAL 后运行对比。


### 10.2 七步回顾

| 阶段 | 输入和输出 | 核对什么 |
|---|---|---|
| 加载 | 文件 → Document | 文本、结构和出处有没有保留 |
| 清洗 | 原始正文 → 规范化正文 | 数字、单位、专名有没有被破坏 |
| 切块 | 正文 → 检索片段 | 证据是否完整，块是否可追溯 |
| 嵌入 | 文本 → 向量 | 建库和查询的模型、维度和配置是否配套 |
| 入库 | 向量、原文、ID、元数据 → 集合 | 重跑会不会重复，过期块有没有清掉 |
| 检索 | 查询 → Top-k | 距离语义是否读对，证据排在哪里 |
| 生成 | 问题和证据 → 答案 | 回答是否完整、引用是否支持、缺证据时怎么答 |

### 按问题定位下一步

| 现象 | 从哪里查 |
|---|---|
| 有答案，但从索引里查不到 | 先核对原文和切块是否保留了证据，再检查召回 |
| 专名或编号经常遗漏 | 在评测里对比关键词、元数据过滤和混合检索 |
| 证据进了候选，但排名靠后 | 固定候选规模，比较重排前后的证据覆盖 |
| 证据已进上下文，答案仍遗漏或引错 | 对照实际传给模型的片段，核对生成和引用 |

### 待办

1. 扩充小评测集，补充同义问法、多个等价出处和易混淆的专名。
2. 复用这批题比较 BM25 与向量混合检索，以及粗召回加重排。
3. 加入父子块方案，记录子块召回后回填父块对证据覆盖和上下文长度的影响。
4. 换成有真实版面结构的 PDF，抽样对比按页抽取与结构化解析。

## 自测题

1. 为什么整篇输入和逐段输入会得到不同数量的检索块？
2. 启用正则后，课堂的分隔符为何仍没有命中中文句界？
3. 为什么 `chunk_overlap=60` 有时实际重叠为 0？
4. Chroma 的 cosine 距离越大还是越小越接近？什么时候能与平方 L2 直接换算？
5. 门票题和「第十九个景点」分别如何区分缺证据和漏召回？
6. 第 8 节如何知道模型实际看到了什么？第 9 节如何把出处传进工具结果？
7. 对比两种切法时，为什么要固定模型、查询和 `k`，同时观察上下文长度？

## 复跑步骤

1. 选择 `week23` 的 Python 3.11 内核，从 `week25` 运行。若从其他目录启动，在第 0 节填写 `DATA_ROOT`。
2. 按第 0 节打印的依赖补齐缺失项。从头运行一次，确认 Word、PDF、清洗、切块和证据校验。
3. 要跑云端部分，在 `week23/.env` 配置 `LLM_PROVIDER=qwen` 和对应的模型凭据。在第 0 节开启 `RUN_LIVE`，重启内核后从头运行。
4. 要对比两种切法，同时开启 `RUN_EVAL`。它会再给整篇递归切块建立一个内存集合，并发起评测请求。

| 现象 | 排查 |
|---|---|
| 找不到讲解词或配置 | 确认 `DATA_ROOT` 指向 `week25` 或材料目录，且 `week23` 在同一课程根目录下 |
| 嵌入批量请求报错 | 检查当前供应商的批量上限和输入长度，本讲 v4 每批 10 条 |
| 重跑后集合条数不符 | 检查落盘目录、集合名称和记录 ID，确认有没有复用旧集合 |
| 云端接口拒绝请求 | 按实际错误检查鉴权、服务端点、模型权限或额度 |